# 8-Puzzle Solver
Choose between BFS, DFS, UCS, DLS, IDS, GBFS, and A*

Please run cells in order or run all.


In [ ]:
from collections import deque
from copy import deepcopy
from heapq import heappush, heappop

# Define start state

start_state = [[5,4,0],
               [6,1,8],
               [7,3,2]]

'''
start_state = [[4,5,8],
               [3,1,0],
               [7,6,2]]
'''

# Define the goal state
goal_state = [[1,2,3],
              [7,8,0],
              [4,5,6]]

**HELPER FUNCTIONS**

In [ ]:
# Asks for search algo
def what_search():
    while True:
        print("0. Exit")
        print("1. Breadth-First Search        (BFS)")
        print("2. Depth-First Search          (DFS)")
        print("3. Uniform Cost Search         (UCS)")
        print("4. Depth-Limited Search        (DLS)")
        print("5. Iterative Deepening Search  (IDS)")
        print("6. Greedy Best-First Search    (GBFS)")
        print("7. A* Search")
        try:
            choice = int(input("Enter your chosen search algorithm (0-7): "))
            if choice == 0:
                print("Exiting program.")
                exit(0)
            elif 1 <= choice <= 7:
                return choice
            else:
                print("Invalid choice. Please enter a number between 0 to 7.")
        except ValueError:
            print("Invalid input. Please enter an integer between 0 and 7.")

In [ ]:
# Helper function to find the position of the blank (0)
def find_blank(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j

# Check if two states are equal
def is_goal(state):
    return state == goal_state

# Generate new states from the current state
def get_neighbors(state):
    neighbors = []
    x, y = find_blank(state)
    moves = [(-1,0),(1,0),(0,-1),(0,1)]  # up, down, left, right

    for dx, dy in moves:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            new_state = [row[:] for row in state]  # deep copy
            new_state[x][y],new_state[nx][ny] =new_state[nx][ny],new_state[x][y]
            neighbors.append(new_state)
    return neighbors

# Convert state to a tuple for hashing
def state_to_tuple(state):
    return tuple(tuple(row) for row in state)


**IMPLEMENTATION OF SEARCHING ALGORITHMS**

I. Uninformed Search

In [ ]:
# BFS Implementation
def bfs(start_state):
    visited = set()
    queue = deque()
    queue.append((start_state, []))  # state and path
    visited.add(state_to_tuple(start_state))

    while queue:
        state, path = queue.popleft()

        if is_goal(state):
            return path + [state]

        for neighbor in get_neighbors(state):
            t_neighbor = state_to_tuple(neighbor)
            if t_neighbor not in visited:
                visited.add(t_neighbor)
                queue.append((neighbor, path + [state]))

    return None  # if no solution is found

In [ ]:
# DFS Implementation
def dfs(start_state, max_depth=50):
    visited = set()
    stack = [(start_state, [], 0)] #state,path,depth

    while stack:
        state, path, depth = stack.pop()

        if is_goal(state):
            return path + [state]

        if depth >= max_depth:
            continue

        t_state = state_to_tuple(state)
        if t_state not in visited:
            visited.add(t_state)
            #reverse to simulate left-to-right exploration
            for neighbor in reversed(get_neighbors(state)):
                stack.append((neighbor, path + [state], depth + 1))

    return None  # if no solution found

In [ ]:
# UCS Implementation
def ucs(start_state):
    visited = set()
    pq = []
    heappush(pq, (0, start_state, []))#(cost, state, path)

    while pq:
        cost, state, path = heappop(pq)

        if is_goal(state):
            return path + [state], cost

        t_state = state_to_tuple(state)
        if t_state not in visited:
            visited.add(t_state)

            for neighbor in get_neighbors(state):
                heappush(pq, (cost+1, neighbor, path+[state]))#cost per move=1

    return None, None  # no solution found

In [ ]:
# DLS Implementation
def dls(start_state, limit, visited=None):
    if visited is None:
        visited = set()

    stack = [(start_state, [], 0)] #state, path, depth

    while stack:
        state, path, depth = stack.pop()

        if is_goal(state):
            return path + [state]

        if depth >= limit:
            continue

        t_state = state_to_tuple(state)
        if t_state not in visited:
            visited.add(t_state)

            for neighbor in reversed(get_neighbors(state)):
                stack.append((neighbor, path + [state], depth + 1))

    return None  # no solution within depth limit

In [ ]:
# IDS Implementation
def ids(start_state, max_depth=50):
    for depth in range(max_depth):
        visited = set()
        result = dls(start_state, depth, visited)

        if result is not None:
            return result

    return None  # no solution

II. Informed Search

In [ ]:
# Heuristic: Manhattan Distance for GBFS and A*
def manhattan_distance(state):
    distance = 0
    for i in range(3):
        for j in range(3):
            val = state[i][j]
            if val != 0:
                goal_x, goal_y = divmod(val - 1, 3)
                distance += abs(i - goal_x) + abs(j - goal_y)
    return distance

In [ ]:
# GBFS Implementation
def gbfs(start_state):
    visited = set()
    pq = []  # (heuristic, state, path)
    heappush(pq, (manhattan_distance(start_state), start_state, []))
    move = 0 # to determine expanding node values

    while pq:
        h, state, path = heappop(pq)
        move += 1

        #4. For GBFS,what are the node values for 20-25th move h(n)?
        if 20 <= move <=25:
            print(f"Move {move}: h(n)={h}")

        if is_goal(state):
            return path + [state]

        t_state = state_to_tuple(state)
        if t_state not in visited:
            visited.add(t_state)
            for neighbor in get_neighbors(state):
                h_val = manhattan_distance(neighbor)
                heappush(pq, (h_val, neighbor, path + [state]))

    return None  # no solution

In [ ]:
# A* Implementation
def a_star(start_state):
    visited = set()
    pq = []
    g_cost = 0
    h_cost = manhattan_distance(start_state)
    f_cost = g_cost + h_cost
    heappush(pq, (f_cost, g_cost, start_state, []))  # (f, g, state, path)

    move = 0 # to determine expanding node values

    while pq:
        f, g, state, path = heappop(pq)
        move += 1

        # calculate h for current state
        h = f - g

        #5. For A*, what are node values for the 10th move g=?,h=?,f=?
        if move == 10:
            print(f"Move 10: g={g}, h={h}, f={f}\n")

        # display expansion details
        if is_goal(state):
            return path + [state], g  # solution and number of moves

        t_state = state_to_tuple(state)
        if t_state not in visited:
            visited.add(t_state)
            for neighbor in get_neighbors(state):
                g_new = g + 1
                h_new = manhattan_distance(neighbor)
                f_new = g_new + h_new
                heappush(pq, (f_new, g_new, neighbor, path + [state]))

    return None, None  # no solution

In [ ]:
# Solve puzzle (run this cell)
choice = what_search()
solution = None
total_cost = None
total_moves = None
depth_limit = 30 #adjust as needed
max_depth = 100 #adjust as needed

if choice == 1:
    print("Breadth-First Search Algorithm\n")
    solution = bfs(start_state)
elif choice == 2:
    print("Depth-First Search Algorithm\n")
    solution = dfs(start_state)
elif choice == 3:
    print("Uniform Cost Search Algorithm\n")
    solution, total_cost = ucs(start_state)
elif choice == 4:
    print("Depth-Limited Search Algorithm\n")
    print(f"Depth limit = {depth_limit}")
    solution = dls(start_state, depth_limit)
elif choice == 5:
    print("Iterative Deepening Search Algorithm\n")
    solution = ids(start_state, max_depth)
elif choice == 6:
    print("Greedy Best-First Search Algorithm\n")
    solution = gbfs(start_state)
elif choice == 7:
    print("A* Search Algorithm\n")
    solution, total_moves = a_star(start_state)

# Display the solution
if solution:
    if choice == 7: #For A-star
        print(f"Solution found in {total_moves} moves:")
    else:
        print(f"Solution found in {len(solution)-1} moves:")

    #1. How many total steps for BFS,DFS,IDS?
    #3. For DLS, where depth limit=30, what is total steps?
    if choice in [1,2,4,5]: # BFS,DFS,DLS,IDS
        print(f"Total steps = {len(solution)-1}")

    #2. For UCS where cost per move=1, what is total cost?
    if choice == 3:
        print(f"Total cost = {total_cost}")

    for step, state in enumerate(solution):
        print(f"Step {step}:")
        for row in state:
            print(row)
        print()
else:
    print("No solution found", end="")
    if choice in (2, 4, 5):
        print(" within depth limit", end="")
    print(".")

0. Exit
1. Breadth-First Search        (BFS)
2. Depth-First Search          (DFS)
3. Uniform Cost Search         (UCS)
4. Depth-Limited Search        (DLS)
5. Iterative Deepening Search  (IDS)
6. Greedy Best-First Search    (GBFS)
7. A* Search
Enter your chosen search algorithm (0-7): 6
Greedy Best-First Search Algorithm

Move 20: h(n)=10
Move 21: h(n)=10
Move 22: h(n)=9
Move 23: h(n)=8
Move 24: h(n)=7
Move 25: h(n)=6
Solution found in 99 moves:
Step 0:
[5, 4, 0]
[6, 1, 8]
[7, 3, 2]

Step 1:
[5, 0, 4]
[6, 1, 8]
[7, 3, 2]

Step 2:
[5, 1, 4]
[6, 0, 8]
[7, 3, 2]

Step 3:
[5, 1, 4]
[0, 6, 8]
[7, 3, 2]

Step 4:
[0, 1, 4]
[5, 6, 8]
[7, 3, 2]

Step 5:
[1, 0, 4]
[5, 6, 8]
[7, 3, 2]

Step 6:
[1, 4, 0]
[5, 6, 8]
[7, 3, 2]

Step 7:
[1, 4, 8]
[5, 6, 0]
[7, 3, 2]

Step 8:
[1, 4, 8]
[5, 0, 6]
[7, 3, 2]

Step 9:
[1, 4, 8]
[5, 3, 6]
[7, 0, 2]

Step 10:
[1, 4, 8]
[5, 3, 6]
[7, 2, 0]

Step 11:
[1, 4, 8]
[5, 3, 0]
[7, 2, 6]

Step 12:
[1, 4, 8]
[5, 0, 3]
[7, 2, 6]

Step 13:
[1, 0, 8]
[5, 4, 3]
[7, 2, 6]

